# Task 2: Sequential Pattern Mining for Cancer Diagnosis

**Objective:** Apply GSP algorithm to discover sequential patterns in breast cancer features.

**Approach:**
- Percentile-based feature ranking
- GSP implementation
- Comprehensive sensitivity analysis

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, KBinsDiscretizer
from scipy.stats import percentileofscore
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

## Step 1: Data Loading and Preparation

In [2]:
# Load cancer dataset
cancer_data = pd.read_csv('Cancer_Data.csv')

# Remove unnecessary columns
cancer_data = cancer_data.drop(['id', 'Unnamed: 32'], axis=1, errors='ignore')

# Extract features and labels
X_features = cancer_data.drop('diagnosis', axis=1)
y_diagnosis = cancer_data['diagnosis']

print(f"Dataset shape: {X_features.shape}")
print(f"\nClass distribution:")
print(y_diagnosis.value_counts())
print(f"\nSample features: {list(X_features.columns[:5])}...")

Dataset shape: (569, 30)

Class distribution:
diagnosis
B    357
M    212
Name: count, dtype: int64

Sample features: ['radius_mean', 'texture_mean', 'perimeter_mean', 'area_mean', 'smoothness_mean']...


## Step 2: Feature Standardization

In [3]:
# Apply z-score standardization
standardizer = StandardScaler()
X_standardized = standardizer.fit_transform(X_features)

# Convert to dataframe for easier manipulation
df_standardized = pd.DataFrame(
    X_standardized,
    columns=X_features.columns,
    index=X_features.index
)

print(f"Standardized data shape: {df_standardized.shape}")
print(f"Mean after standardization: {df_standardized.mean().mean():.6f}")
print(f"Std after standardization: {df_standardized.std().mean():.6f}")

Standardized data shape: (569, 30)
Mean after standardization: -0.000000
Std after standardization: 1.000880


## Step 3: Multi-Strategy Discretization

In [4]:
class FeatureDiscretizer:
    """
    Handles feature discretization with multiple strategies.
    Encapsulates discretization logic in a class.
    """

    def __init__(self, num_bins=3):
        self.num_bins = num_bins
        self.bin_labels = ['low', 'mid', 'high'][:num_bins]
        self.discretizers = {}

    def fit_transform(self, data, strategy):
        """
        Apply discretization using specified strategy.

        Args:
            data: Feature dataframe
            strategy: 'uniform', 'quantile', or 'kmeans'

        Returns:
            Discretized dataframe with string labels
        """
        # Initialize discretizer for this strategy
        discretizer = KBinsDiscretizer(
            n_bins=self.num_bins,
            encode='ordinal',
            strategy=strategy,
            subsample=None
        )

        # Fit and transform
        binned_values = discretizer.fit_transform(data)

        # Store discretizer
        self.discretizers[strategy] = discretizer

        # Create labeled dataframe
        df_binned = pd.DataFrame(
            binned_values,
            columns=data.columns,
            index=data.index
        )

        # Convert numeric bins to string labels
        mapping = {float(i): label for i, label in enumerate(self.bin_labels)}
        df_labeled = df_binned.replace(mapping)

        return df_labeled

    def discretize_all_strategies(self, data):
        """
        Apply all three strategies and return dictionary.
        """
        strategies = ['uniform', 'quantile', 'kmeans']
        results = {}

        for strategy in strategies:
            results[strategy] = self.fit_transform(data, strategy)

        return results

# Create discretizer and apply all strategies
discretizer = FeatureDiscretizer(num_bins=3)
discretized_datasets = discretizer.discretize_all_strategies(X_features)

print("\nSample discretized data (quantile):")
print(discretized_datasets['quantile'].head(3))


Sample discretized data (quantile):
  radius_mean texture_mean perimeter_mean area_mean smoothness_mean  \
0        high          low           high      high            high   
1        high          mid           high      high             low   
2        high         high           high      high            high   

  compactness_mean concavity_mean concave points_mean symmetry_mean  \
0             high           high                high          high   
1              mid            mid                high           mid   
2             high           high                high          high   

  fractal_dimension_mean  ... radius_worst texture_worst perimeter_worst  \
0                   high  ...         high           low            high   
1                    low  ...         high           mid            high   
2                    mid  ...         high           mid            high   

  area_worst smoothness_worst compactness_worst concavity_worst  \
0       high         

## Step 4: Sequence Builder - Percentile Ranking Approach

In [5]:
class PatientSequenceBuilder:
    """
    Builds sequential representations of patient features.
    Uses percentile-based ranking for feature selection.
    """

    def __init__(self, top_k=10, grouping_threshold=1e-5):
        self.top_k = top_k
        self.grouping_threshold = grouping_threshold

    def calculate_percentile_ranks(self, patient_values):
        """
        Calculate percentile rank for each feature value.

        Returns:
            DataFrame with features, absolute values, and percentiles
        """
        abs_values = patient_values.abs()

        # Calculate percentile for each value
        percentiles = []
        for val in abs_values.values:
            pct = percentileofscore(abs_values.values, val) / 100.0
            percentiles.append(pct)

        # Create ranking dataframe
        rank_df = pd.DataFrame({
            'feature': abs_values.index,
            'abs_value': abs_values.values,
            'percentile': percentiles
        })

        return rank_df.sort_values('percentile', ascending=False)

    def group_by_percentile(self, ranked_features, discretized_levels):
        """
        Group features with similar percentiles into itemsets.

        Args:
            ranked_features: DataFrame with ranked features
            discretized_levels: Series with discretized levels for this patient

        Returns:
            List of itemsets (list of lists)
        """
        sequence = []
        current_itemset = []
        previous_percentile = None

        for _, row in ranked_features.iterrows():
            feature = row['feature']
            percentile = row['percentile']

            # Get discretized level
            level = discretized_levels[feature]
            token = f"{feature}={level}"

            # Decide if we start a new itemset
            if previous_percentile is None:
                current_itemset = [token]
            elif abs(percentile - previous_percentile) < self.grouping_threshold:
                current_itemset.append(token)
            else:
                sequence.append(current_itemset)
                current_itemset = [token]

            previous_percentile = percentile

        # Add last itemset
        if current_itemset:
            sequence.append(current_itemset)

        return sequence

    def build_sequence(self, patient_id, standardized_data, discretized_data):
        """
        Build complete sequence for a patient.

        Returns:
            List of itemsets representing feature sequence
        """
        # Get patient's standardized values
        patient_values = standardized_data.loc[patient_id]

        # Calculate percentile ranks
        ranked = self.calculate_percentile_ranks(patient_values)

        # Select top k features
        top_features = ranked.head(self.top_k)

        # Get discretized levels for this patient
        patient_levels = discretized_data.loc[patient_id]

        # Group into itemsets
        sequence = self.group_by_percentile(top_features, patient_levels)

        return sequence

# Test the sequence builder
seq_builder = PatientSequenceBuilder(top_k=10, grouping_threshold=1e-5)
test_sequence = seq_builder.build_sequence(
    patient_id=0,
    standardized_data=df_standardized,
    discretized_data=discretized_datasets['quantile']
)

print("Sample patient sequence:")
for i, itemset in enumerate(test_sequence, 1):
    print(f"  Itemset {i}: {itemset}")

Sample patient sequence:
  Itemset 1: ['compactness_mean=high']
  Itemset 2: ['perimeter_se=high']
  Itemset 3: ['symmetry_worst=high']
  Itemset 4: ['concavity_mean=high']
  Itemset 5: ['compactness_worst=high']
  Itemset 6: ['concave points_mean=high']
  Itemset 7: ['radius_se=high']
  Itemset 8: ['area_se=high']
  Itemset 9: ['perimeter_worst=high']
  Itemset 10: ['concave points_worst=high']


## Step 5: Generate Sequences for All Patients

In [6]:
def generate_all_patient_sequences(standardized_data, discretized_data, diagnoses, strategy_name):
    """
    Generate sequences for all patients using specified discretization.

    Returns:
        Tuple of (sequences_list, diagnoses_list)
    """
    builder = PatientSequenceBuilder(top_k=5, grouping_threshold=1e-6)

    sequences = []
    labels = []

    for patient_id in standardized_data.index:
        seq = builder.build_sequence(
            patient_id,
            standardized_data,
            discretized_data[strategy_name]
        )
        sequences.append(seq)
        labels.append(diagnoses.loc[patient_id])

    return sequences, labels

# Generate sequences for all strategies
all_sequences = {}

for strategy in ['uniform', 'quantile', 'kmeans']:
    seqs, labels = generate_all_patient_sequences(
        df_standardized,
        discretized_datasets,
        y_diagnosis,
        strategy
    )
    all_sequences[strategy] = {
        'sequences': seqs,
        'labels': labels
    }
    print(f"{strategy}: {len(seqs)} sequences generated")

uniform: 569 sequences generated
quantile: 569 sequences generated
kmeans: 569 sequences generated


## Step 6: Format Sequences for Mining

In [7]:
class SequenceFormatter:
    """
    Formats sequences for pattern mining.
    Handles conversion from nested itemsets to flat strings.
    """

    @staticmethod
    def format_itemset(itemset):
        if len(itemset) == 1:
            return itemset[0]
        else:
            return '|'.join(sorted(itemset))

    @staticmethod
    def format_sequence(nested_sequence):
        """
        Convert nested sequence to flat list of strings.
        """
        return [SequenceFormatter.format_itemset(itemset)
                for itemset in nested_sequence]

    @staticmethod
    def prepare_database(sequences, labels, target_diagnosis):
        """
        Filter and format sequences for specific diagnosis.
        """
        formatted_db = []

        for seq, label in zip(sequences, labels):
            if label == target_diagnosis:
                formatted_seq = SequenceFormatter.format_sequence(seq)
                formatted_db.append(formatted_seq)

        return formatted_db

# Prepare databases for quantile strategy
formatter = SequenceFormatter()
quantile_seqs = all_sequences['quantile']

sequences_malignant = formatter.prepare_database(
    quantile_seqs['sequences'],
    quantile_seqs['labels'],
    'M'
)

sequences_benign = formatter.prepare_database(
    quantile_seqs['sequences'],
    quantile_seqs['labels'],
    'B'
)

print(f"Malignant sequences: {len(sequences_malignant)}")
print(f"Benign sequences: {len(sequences_benign)}")
print(f"\nSample malignant sequence:\n{sequences_malignant[0]}")
print(f"\nSample benign sequence:\n{sequences_benign[0]}")

Malignant sequences: 212
Benign sequences: 357

Sample malignant sequence:
['compactness_mean=high', 'perimeter_se=high', 'symmetry_worst=high', 'concavity_mean=high', 'compactness_worst=high']

Sample benign sequence:
['texture_mean=low', 'texture_worst=low', 'texture_se=low', 'fractal_dimension_mean=low', 'fractal_dimension_worst=low']


## Step 7: GSP Pattern Mining Algorithm

In [8]:
class GSPMiner:
    """
    Generalized Sequential Pattern (GSP) mining algorithm.
    """

    def __init__(self, min_support_ratio=0.1, max_pattern_length=3):
        self.min_support_ratio = min_support_ratio
        self.max_pattern_length = max_pattern_length
        self.discovered_patterns = []

    def calculate_min_support(self, database_size):
        """
        Calculate minimum support count from ratio.
        """
        return max(1, int(np.ceil(self.min_support_ratio * database_size)))

    def is_subsequence(self, sequence, pattern):
        """
        Check if pattern is an ordered subsequence of sequence.
        """
        pattern_idx = 0

        for item in sequence:
            if pattern_idx < len(pattern) and item == pattern[pattern_idx]:
                pattern_idx += 1

        return pattern_idx == len(pattern)

    def count_support(self, database, pattern):
        """
        Count how many sequences contain the pattern.
        """
        support_count = 0
        for sequence in database:
            if self.is_subsequence(sequence, pattern):
                support_count += 1
        return support_count

    def find_frequent_1_sequences(self, database, min_support):
        """
        Phase 1: Find all frequent 1-item sequences.
        """
        # Count item occurrences
        item_counts = defaultdict(int)

        for sequence in database:
            unique_items = set(sequence)
            for item in unique_items:
                item_counts[item] += 1

        # Filter by minimum support
        frequent_items = {}
        for item, count in item_counts.items():
            if count >= min_support:
                pattern = (item,)
                frequent_items[pattern] = count
                self.discovered_patterns.append((count, list(pattern)))

        return frequent_items

    def generate_candidates(self, frequent_patterns, frequent_1_items):
        """
        Generate candidate patterns by extending existing patterns.
        """
        candidates = set()

        # Extend each frequent pattern with each frequent 1-item
        for pattern in frequent_patterns:
            for item_pattern in frequent_1_items:
                # Create new candidate by appending
                new_candidate = pattern + item_pattern
                candidates.add(new_candidate)

        return candidates

    def prune_candidates(self, candidates, previous_frequent, pattern_length):
        """
        Prune candidates using Apriori property.
        All (k-1)-subsequences must be frequent.
        """
        pruned = set()

        for candidate in candidates:
            # Check all (k-1)-length subsequences
            is_valid = True

            for i in range(len(candidate)):
                # Create subsequence by removing one element
                subsequence = candidate[:i] + candidate[i+1:]

                if subsequence not in previous_frequent:
                    is_valid = False
                    break

            if is_valid:
                pruned.add(candidate)

        return pruned

    def mine_patterns(self, database):
        """
        Main GSP mining procedure.

        Returns:
            List of (support, pattern) tuples sorted by support
        """
        n_sequences = len(database)
        min_support = self.calculate_min_support(n_sequences)

        print(f"Mining with min_support = {min_support}/{n_sequences} "
              f"({self.min_support_ratio:.0%})")

        # Phase 1: Find frequent 1-sequences
        frequent_1 = self.find_frequent_1_sequences(database, min_support)
        print(f"  Level 1: Found {len(frequent_1)} frequent items")

        # Store frequent patterns from previous level
        previous_level_patterns = set(frequent_1.keys())
        frequent_1_items = set(frequent_1.keys())

        # Iteratively find longer patterns
        for length in range(2, self.max_pattern_length + 1):
            # Generate candidates
            candidates = self.generate_candidates(
                previous_level_patterns,
                frequent_1_items
            )

            # Prune candidates
            candidates = self.prune_candidates(
                candidates,
                previous_level_patterns,
                length
            )

            # Count support for candidates
            current_level_patterns = {}

            for candidate in candidates:
                support = self.count_support(database, candidate)

                if support >= min_support:
                    current_level_patterns[candidate] = support
                    self.discovered_patterns.append((support, list(candidate)))

            print(f"  Level {length}: Found {len(current_level_patterns)} frequent patterns")

            # Stop if no frequent patterns found
            if not current_level_patterns:
                break

            previous_level_patterns = set(current_level_patterns.keys())

        # Sort patterns by support (descending) then by length (ascending)
        self.discovered_patterns.sort(key=lambda x: (-x[0], len(x[1])))

        return self.discovered_patterns

# Mine patterns for malignant cases
print("\n-------------------------------")
print("Mining Malignant Patterns")
print("-------------------------------")
miner_malignant = GSPMiner(min_support_ratio=0.1, max_pattern_length=3)
patterns_malignant = miner_malignant.mine_patterns(sequences_malignant)

# Mine patterns for benign cases
print("\n-------------------------------")
print("Mining Benign Patterns")
print("-------------------------------")
miner_benign = GSPMiner(min_support_ratio=0.1, max_pattern_length=3)
patterns_benign = miner_benign.mine_patterns(sequences_benign)


-------------------------------
Mining Malignant Patterns
-------------------------------
Mining with min_support = 22/212 (10%)
  Level 1: Found 22 frequent items
  Level 2: Found 3 frequent patterns
  Level 3: Found 0 frequent patterns

-------------------------------
Mining Benign Patterns
-------------------------------
Mining with min_support = 36/357 (10%)
  Level 1: Found 22 frequent items
  Level 2: Found 1 frequent patterns
  Level 3: Found 0 frequent patterns


## Step 8: Display Discovered Patterns

In [9]:
def display_pattern_results(patterns, database_size, diagnosis_label, top_n=None):
    """
    Display patterns in formatted output.
    - If top_n is None (default): Displays ALL patterns.
    - If top_n is a number: Displays that many top patterns.
    """
    if top_n is None:
        patterns_to_show = patterns
        title = f"All {len(patterns)} Patterns: {diagnosis_label}"
    else:
        top_n_actual = min(top_n, len(patterns))
        patterns_to_show = patterns[:top_n_actual]
        title = f"Top {top_n_actual} Patterns: {diagnosis_label}"

    print("\n-------------------------------")
    print(title)
    print("-------------------------------")

    if not patterns:
        print(f"No patterns found for {diagnosis_label}.")
        return
    if not patterns_to_show:
        print(f"No patterns to display for {diagnosis_label}.")
        return

    max_rank_len = len(str(len(patterns_to_show)))
    indent_space = " " * (max_rank_len + 1)

    for rank, (support, pattern) in enumerate(patterns_to_show, 1):
        coverage = support / database_size
        pattern_display = ' → '.join(pattern)

        print(f"{rank:{max_rank_len}d}. {pattern_display}")
        print(f"{indent_space} Support: {support}/{database_size} ({coverage:.1%})\n")

display_pattern_results(
    patterns_malignant,
    len(sequences_malignant),
    "MALIGNANT CASES",
    top_n=None
)

display_pattern_results(
    patterns_benign,
    len(sequences_benign),
    "BENIGN CASES",
    top_n=None
)


-------------------------------
All 25 Patterns: MALIGNANT CASES
-------------------------------
 1. radius_mean=high
    Support: 60/212 (28.3%)

 2. radius_worst=high
    Support: 49/212 (23.1%)

 3. area_mean=high
    Support: 47/212 (22.2%)

 4. compactness_worst=high
    Support: 45/212 (21.2%)

 5. perimeter_mean=high
    Support: 43/212 (20.3%)

 6. fractal_dimension_worst=high
    Support: 42/212 (19.8%)

 7. texture_worst=high
    Support: 41/212 (19.3%)

 8. perimeter_worst=high
    Support: 40/212 (18.9%)

 9. concave points_mean=high
    Support: 39/212 (18.4%)

10. concavity_worst=high
    Support: 39/212 (18.4%)

11. symmetry_worst=high
    Support: 38/212 (17.9%)

12. area_worst=high
    Support: 38/212 (17.9%)

13. radius_se=high
    Support: 37/212 (17.5%)

14. concave points_worst=high
    Support: 36/212 (17.0%)

15. concavity_mean=high
    Support: 35/212 (16.5%)

16. smoothness_worst=high
    Support: 35/212 (16.5%)

17. perimeter_se=high
    Support: 31/212 (14.6

## Step 9: Sensitivity Analysis

In [10]:
class SensitivityAnalyzer:
    """
    Performs sensitivity analysis across different binning strategies.
    """

    def __init__(self, all_sequences_dict, min_support_ratio=0.1, max_length=3):
        self.all_sequences = all_sequences_dict
        self.min_support_ratio = min_support_ratio
        self.max_length = max_length
        self.results = {}

    def mine_for_strategy(self, strategy_name):
        """
        Mine patterns for a specific binning strategy.
        """
        print("\n-------------------------------")
        print(f"Strategy: {strategy_name.upper()}")
        print("-------------------------------")
        # Get sequences for this strategy
        strategy_data = self.all_sequences[strategy_name]

        # Prepare databases
        formatter = SequenceFormatter()
        db_m = formatter.prepare_database(
            strategy_data['sequences'],
            strategy_data['labels'],
            'M'
        )
        db_b = formatter.prepare_database(
            strategy_data['sequences'],
            strategy_data['labels'],
            'B'
        )

        # Mine malignant patterns
        print(f"Malignant (n={len(db_m)}):")
        miner_m = GSPMiner(self.min_support_ratio, self.max_length)
        patterns_m = miner_m.mine_patterns(db_m)

        # Mine benign patterns
        print(f"Benign (n={len(db_b)}):")
        miner_b = GSPMiner(self.min_support_ratio, self.max_length)
        patterns_b = miner_b.mine_patterns(db_b)

        # Store results
        self.results[strategy_name] = {
            'malignant': {'patterns': patterns_m, 'n_sequences': len(db_m)},
            'benign': {'patterns': patterns_b, 'n_sequences': len(db_b)}
        }

    def run_full_analysis(self):
        """
        Run mining for all three strategies.
        """
        for strategy in ['quantile', 'uniform', 'kmeans']:
            self.mine_for_strategy(strategy)

        return self.results

    def compare_strategies(self, diagnosis_type, top_n=5):
        """
        Compare patterns across strategies for specific diagnosis.
        """
        print(f"\nCOMPARISON: {diagnosis_type.upper()} Patterns Across Strategies")

        for strategy in ['quantile', 'uniform', 'kmeans']:
            print(f"\n{strategy.upper()}:")
            print("-" * 60)

            data = self.results[strategy][diagnosis_type]
            patterns = data['patterns']
            n_seq = data['n_sequences']

            for i, (support, pattern) in enumerate(patterns[:top_n], 1):
                ratio = support / n_seq
                pattern_str = ' → '.join(pattern)
                print(f"  {i}. {pattern_str[:45]:<45} | "
                      f"{support:3d}/{n_seq:3d} ({ratio:5.1%})")

# Run sensitivity analysis
analyzer = SensitivityAnalyzer(
    all_sequences,
    min_support_ratio=0.1,
    max_length=3
)

sensitivity_results = analyzer.run_full_analysis()


-------------------------------
Strategy: QUANTILE
-------------------------------
Malignant (n=212):
Mining with min_support = 22/212 (10%)
  Level 1: Found 22 frequent items
  Level 2: Found 3 frequent patterns
  Level 3: Found 0 frequent patterns
Benign (n=357):
Mining with min_support = 36/357 (10%)
  Level 1: Found 22 frequent items
  Level 2: Found 1 frequent patterns
  Level 3: Found 0 frequent patterns

-------------------------------
Strategy: UNIFORM
-------------------------------
Malignant (n=212):
Mining with min_support = 22/212 (10%)
  Level 1: Found 18 frequent items
  Level 2: Found 1 frequent patterns
  Level 3: Found 0 frequent patterns
Benign (n=357):
Mining with min_support = 36/357 (10%)
  Level 1: Found 19 frequent items
  Level 2: Found 1 frequent patterns
  Level 3: Found 0 frequent patterns

-------------------------------
Strategy: KMEANS
-------------------------------
Malignant (n=212):
Mining with min_support = 22/212 (10%)
  Level 1: Found 20 frequent it

## Step 10: Strategy Comparison

In [11]:
# Compare malignant patterns
analyzer.compare_strategies('malignant', top_n=5)

# Compare benign patterns
analyzer.compare_strategies('benign', top_n=5)


COMPARISON: MALIGNANT Patterns Across Strategies

QUANTILE:
------------------------------------------------------------
  1. radius_mean=high                              |  60/212 (28.3%)
  2. radius_worst=high                             |  49/212 (23.1%)
  3. area_mean=high                                |  47/212 (22.2%)
  4. compactness_worst=high                        |  45/212 (21.2%)
  5. perimeter_mean=high                           |  43/212 (20.3%)

UNIFORM:
------------------------------------------------------------
  1. radius_mean=mid                               |  50/212 (23.6%)
  2. area_mean=mid                                 |  40/212 (18.9%)
  3. radius_worst=mid                              |  38/212 (17.9%)
  4. compactness_worst=mid                         |  35/212 (16.5%)
  5. perimeter_mean=mid                            |  35/212 (16.5%)

KMEANS:
------------------------------------------------------------
  1. radius_mean=high                          